# Arabic LLM Fine-Tuning Hands-On Tutorial

## Complete Guide to Fine-Tuning LLMs for Arabic Language

In this notebook, you will:
1. Learn about Arabic NLP challenges
2. Prepare Arabic datasets for fine-tuning
3. Fine-tune an Arabic LLM using QLoRA
4. Evaluate your fine-tuned model
5. Build an Arabic chatbot

**Prerequisites:**
- GPU with at least 16GB VRAM (24GB+ recommended)
- Python 3.10+
- Basic knowledge of PyTorch and transformers

**Estimated Time:** 2-3 hours

## Step 1: Environment Setup

Install required libraries:

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes
!pip install sentence-transformers camel-tools
!pip install torch --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Verify GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Understanding Arabic NLP Challenges

In [ ]:
import re
import unicodedata

class ArabicTextNormalizer:
    """Normalize Arabic text for consistent processing"""
    
    def normalize(self, text: str) -> str:
        # Alif normalization (أ, إ, آ → ا)
        text = re.sub("[إأآا]", "ا", text)
        
        # Alif Maqsura (ى → ي)
        text = re.sub("ى", "ي", text)
        
        # Ta Marbuta (ة → ه)
        text = re.sub("ة", "ه", text)
        
        # Hamza variations
        text = re.sub("ؤ", "ء", text)
        text = re.sub("ئ", "ء", text)
        
        # Remove diacritics (tashkeel)
        text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
        
        # Unicode normalization
        text = unicodedata.normalize("NFKC", text)
        
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text

# Example
normalizer = ArabicTextNormalizer()

original = "اللُّغَةُ العَرَبِيَّةُ جميلةٌ جداً"
normalized = normalizer.normalize(original)

print(f"Original:  {original}")
print(f"Normalized: {normalized}")

## Step 3: Load and Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load Arabic instruction dataset
# Using OpenAssistant Arabic conversations
dataset = load_dataset("OpenAssistant/oasst1", split="train")

# Filter for Arabic messages
def is_arabic(text):
    """Simple heuristic to detect Arabic text"""
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    return arabic_chars / len(text) > 0.5 if text else False

arabic_dataset = dataset.filter(lambda x: is_arabic(x.get('text', '')))

print(f"Total Arabic samples: {len(arabic_dataset)}")
print(f"\nSample:")
print(arabic_dataset[0]['text'][:200])

In [ ]:
# Create chat-format dataset
def format_as_chat(example):
    """Format as chat conversation"""
    text = example['text']
    message_id = example['message_id']
    
    return {
        "text": f"المستخدم: {text}\nالمساعد:",
        "message_id": message_id
    }

chat_dataset = arabic_dataset.map(format_as_chat, remove_columns=arabic_dataset.column_names)
print(f"Chat dataset size: {len(chat_dataset)}")

## Step 4: Load Model with QLoRA Configuration

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Model selection
MODEL_NAME = "inceptionai/jais-13b-chat"  # Best Arabic LLM
# Alternative: "aubmindlab/bert-base-arabertv2" for encoder tasks

print(f"Loading model: {MODEL_NAME}")

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

In [ ]:
# Configure LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,  # Rank: higher = more expressive
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 5: Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    """Tokenize with normalization"""
    normalizer = ArabicTextNormalizer()
    
    # Normalize Arabic text
    normalized_texts = [normalizer.normalize(text) for text in examples["text"]]
    
    # Tokenize
    tokenized = tokenizer(
        normalized_texts,
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    
    return tokenized

# Tokenize dataset
tokenized_dataset = chat_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=chat_dataset.column_names
)

print(f"Tokenized dataset size: {len(tokenized_dataset)}")
print(f"Sample input shape: {tokenized_dataset['input_ids'][0].shape}")

## Step 6: Training Configuration

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Training arguments optimized for Arabic
training_args = TrainingArguments(
    output_dir="./arabic-chatbot-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-4,  # Slightly higher for Arabic
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    bf16=True,  # Use bfloat16 for stability
    gradient_accumulation_steps=8,  # Effective batch size = 32
    report_to="none",
    push_to_hub=False
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

## Step 7: Fine-Tune the Model

In [ ]:
print("Starting Arabic LLM fine-tuning...")
print(f"Training samples: {len(tokenized_dataset)}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print("-" * 60)

# Train the model
trainer.train()

# Save the fine-tuned model
print("\nSaving model...")
trainer.save_model("./arabic-chatbot-finetuned")
tokenizer.save_pretrained("./arabic-chatbot-finetuned")

print("✓ Fine-tuning complete!")

## Step 8: Test the Fine-Tuned Model

In [ ]:
def generate_arabic_response(model, tokenizer, prompt: str, max_tokens: int = 256) -> str:
    """Generate Arabic response"""
    
    # Normalize prompt
    normalizer = ArabicTextNormalizer()
    normalized_prompt = normalizer.normalize(prompt)
    
    # Tokenize
    inputs = tokenizer(normalized_prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return generated_text

# Test prompts
test_prompts = [
    "ما هو الذكاء الاصطناعي؟",
    "كيف حالك؟",
    "اشرح لي التعلم الآلي",
    "ما هي عاصمة مصر؟"
]

print("Testing Fine-Tuned Arabic Model")
print("=" * 60)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    response = generate_arabic_response(model, tokenizer, prompt)
    print(f"Response: {response}\n")
    print("-" * 60)

## Step 9: Build Arabic Chatbot

In [ ]:
class ArabicChatbot:
    """Simple Arabic chatbot using fine-tuned model"""
    
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.normalizer = ArabicTextNormalizer()
        self.conversation_history = []
        
        self.system_prompt = """أنت مساعد ذكي ومتحدث باللغة العربية الفصحى.
تجيب على الأسئلة بدقة ووضوح.
إذا سُئلت عن شيء لا تعرفه، اعتذر بلباقة."""
    
    def chat(self, user_message: str, max_tokens: int = 512) -> str:
        """Chat with the model"""
        
        # Normalize
        normalized_message = self.normalizer.normalize(user_message)
        
        # Add to history
        self.conversation_history.append(f"المستخدم: {normalized_message}")
        
        # Build prompt
        prompt = f"{self.system_prompt}\n\n" + "\n".join(self.conversation_history) + "\nالمساعد:"
        
        # Tokenize
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.2,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode
        full_response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        assistant_response = full_response.split("المساعد:")[-1].strip()
        
        # Add to history
        self.conversation_history.append(f"المساعد: {assistant_response}")
        
        return assistant_response
    
    def reset(self):
        """Clear conversation history"""
        self.conversation_history = []

# Create chatbot
chatbot = ArabicChatbot(model, tokenizer)

# Test conversation
print("Arabic Chatbot Test")
print("=" * 60)

messages = [
    "مرحباً، من أنت؟",
    "ما هو الذكاء الاصطناعي؟",
    "كيف يمكنني تعلم البرمجة؟"
]

for message in messages:
    print(f"\nYou: {message}")
    response = chatbot.chat(message)
    print(f"Bot: {response}")

## Step 10: Evaluate Model Performance

In [ ]:
from transformers import pipeline

# Load evaluation pipeline
evaluator = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

# Evaluation questions
eval_questions = [
    {"question": "ما هي فوائد الرياضة؟", "category": "general"},
    {"question": "اشرح نظرية النسبية", "category": "science"},
    {"question": "كيف أتعلم اللغة الإنجليزية؟", "category": "education"},
    {"question": "ما هو تاريخ الحضارة الإسلامية؟", "category": "history"},
]

print("Model Evaluation")
print("=" * 60)

for item in eval_questions:
    question = item["question"]
    category = item["category"]
    
    print(f"\nCategory: {category}")
    print(f"Question: {question}")
    
    response = evaluator(question, max_new_tokens=256, do_sample=True, temperature=0.7)
    print(f"Response: {response[0]['generated_text']}")
    print("-" * 60)

## Summary and Next Steps

### What You Learned:
✓ Arabic text normalization techniques
✓ Loading and preparing Arabic datasets
✓ QLoRA fine-tuning for Arabic LLMs
✓ Building Arabic chatbots
✓ Model evaluation strategies

### Next Steps:
1. **Experiment with different models**: Try Jais-30B, AceGPT, or AraBERT
2. **Use larger datasets**: OSIAN, Arabic Billion Words, OpenAssistant Arabic
3. **Fine-tune for specific tasks**: Sentiment analysis, NER, QA
4. **Optimize hyperparameters**: Learning rate, rank, batch size
5. **Deploy to production**: Use vLLM or TGI for serving

### Resources:
- [CAMeL Tools](https://github.com/CAMeL-Lab/camel_tools) - Arabic NLP toolkit
- [Arabic NLP Datasets](https://huggingface.co/datasets?search=arabic) - Hugging Face
- [Jais Model](https://inceptionai.ai/jais) - Arabic-centric LLM

### Congratulations! 🎉
You've successfully fine-tuned an Arabic LLM!